# 🎬 Supernan AI Intern – Hindi Dubbing Pipeline
**Google Colab Notebook (Free T4 GPU)**

> **Runtime**: Set Runtime → **T4 GPU** before running.
> **Source language**: Kannada (auto-detected). Clip window: **0:45 – 1:00** (confirmed speech).

## Cell 1 – Install System Packages

In [ ]:
!apt-get install -y -q ffmpeg rubberband-cli
!ffmpeg -version | head -1
print('✓ System packages ready')

## Cell 2 – Install Python Dependencies

**Note**: Coqui `TTS` PyPI requires Python <3.12. We use the community fork `coqui-ai-tts`.

In [ ]:
# Core utilities
!pip install -q ffmpeg-python pydub colorlog soundfile pyrubberband numpy
print('✓ Core utilities')

# Whisper ASR
!pip install -q openai-whisper
print('✓ Whisper')

# Translation
!pip install -q deep-translator
print('✓ deep-translator')

# Coqui XTTS v2 — Python 3.12 compatible community fork
!pip install -q git+https://github.com/idiap/coqui-ai-tts
print('✓ Coqui XTTS v2')

# Face restoration
!pip install -q facexlib basicsr gfpgan
print('✓ GFPGAN')

# HuggingFace stack
!pip install -q huggingface_hub transformers sentencepiece sacremoses
print('✓ HuggingFace stack')

# OpenCV
!pip install -q opencv-python-headless
print('✓ OpenCV')

## Cell 3 – Install IndicTrans2 Toolkit

**Fix**: Use the official PyPI package `indictranstoolkit` (not the git repo which has no setup.py).

In [ ]:
!pip install -q indictranstoolkit
print('✓ IndicTransToolkit installed')

## Cell 4 – Clone VideoReTalking

In [ ]:
import os

if not os.path.exists('video-retalking'):
    !git clone https://github.com/vinthony/video-retalking.git

if not os.path.exists('video-retalking/checkpoints'):
    os.makedirs('video-retalking/checkpoints', exist_ok=True)
    from huggingface_hub import snapshot_download
    snapshot_download(
        'vinthony/video-retalking',
        local_dir='video-retalking/checkpoints',
        ignore_patterns=['*.git*'],
    )

# Install VideoReTalking deps (|| true = ignore non-fatal build errors)
!pip install -q -r video-retalking/requirements.txt 2>&1 | tail -3 || true
print('✓ VideoReTalking ready')

## Cell 5 – Clone Pipeline Repo

In [ ]:
REPO_URL = 'https://github.com/sudip-kumar-prasad/supernan-hindi-dubbing-pipeline.git'

if os.path.exists('supernan-hindi-dubbing-pipeline'):
    !git -C supernan-hindi-dubbing-pipeline pull
else:
    !git clone {REPO_URL}

import sys
sys.path.insert(0, '/content/supernan-hindi-dubbing-pipeline')
print('✓ Pipeline repo ready')

## Cell 6 – Download Source Video

In [ ]:
!pip install -q gdown

# Option A: Google Drive
FILE_ID = '1uDzLVEow_gAJsXnNjbSoskzVLZ4d7opW'
!gdown {FILE_ID} -O /content/input.mp4

# Option B: manual upload (if Option A fails)
# from google.colab import files; files.upload()

import os
assert os.path.exists('/content/input.mp4'), 'input.mp4 not found'
print(f'✓ Video ready: {os.path.getsize("/content/input.mp4") / 1e6:.1f} MB')

## Cell 7 – Run the Pipeline

In [ ]:
os.chdir('/content/supernan-hindi-dubbing-pipeline')

# 0:45–1:00 = confirmed Kannada speech. large-v3 gives best Kannada ASR quality.
!python dub_video.py \
    --input /content/input.mp4 \
    --output /content/output.mp4 \
    --start 45 \
    --end 60 \
    --model large-v3 \
    --verbose

## Cell 8 – Preview & Download Output

In [ ]:
from IPython.display import Video, display
import os

output_path = '/content/output.mp4'
assert os.path.exists(output_path), 'output.mp4 not found – check Cell 7 logs'
print(f'Output: {os.path.getsize(output_path) / 1e6:.1f} MB')
display(Video(output_path, embed=True, width=640))

In [ ]:
from google.colab import files
files.download('/content/output.mp4')

---
## 💡 Tips

- Source is **Kannada** — `large-v3` Whisper handles it well; `base` will miss segments.
- If a step crashes, just re-run it — pipeline auto-resumes from last checkpoint.
- Add `--batch` for videos longer than 5 minutes (silence-based audio chunking).
- If VideoReTalking OOMs, pipeline auto-falls back to Wav2Lip.
- For a quick test without GPU stages: add `--skip-lipsync --skip-enhance`.